# RAG From Scratch: GraphRAG and LightRAG

GraphRAG is useful when answers require relationships across documents, not just the nearest chunks. This notebook builds a tiny graph index locally, then shows local and global retrieval patterns.

In [ ]:
import sys
!uv pip install --python {sys.executable} -r pyproject.toml

## Model provider

Shared provider selector for notebook generation cells. `RAG_CHAT_PROVIDER=gemini` keeps the original Gemini chat path. Set `RAG_CHAT_PROVIDER=openrouter` and `OPENROUTER_API_KEY` in `.env` to use OpenRouter cheap models. Retrieval examples default to `RAG_EMBEDDING_PROVIDER=local` for API-free TF-IDF embeddings; set `RAG_EMBEDDING_PROVIDER=gemini` for the original Gemini embeddings.


In [ ]:
from rag_providers import (
    OPENROUTER_MODELS,
    chat_model,
    chat_provider_name,
    embedding_model,
    embedding_provider_name,
    openrouter_model_name,
)

print("Chat provider:", chat_provider_name())
print("Embedding provider:", embedding_provider_name())
print("OpenRouter model:", openrouter_model_name())
OPENROUTER_MODELS


In [ ]:
import itertools
import re
import networkx as nx

docs = [
    {"id": "d1", "text": "GraphRAG extracts entities and relationships from private documents."},
    {"id": "d2", "text": "GraphRAG uses community summaries for global questions."},
    {"id": "d3", "text": "LightRAG combines graph structures with vector retrieval for efficient updates."},
    {"id": "d4", "text": "Hybrid search and reranking remain useful before graph expansion."},
]

def extract_entities(text):
    candidates = re.findall(r"\b[A-Z][A-Za-z0-9-]*\b", text)
    return sorted(set(candidates))

graph = nx.Graph()
for doc in docs:
    entities = extract_entities(doc["text"])
    for entity in entities:
        graph.add_node(entity, docs=set())
        graph.nodes[entity]["docs"].add(doc["id"])
    for left, right in itertools.combinations(entities, 2):
        graph.add_edge(left, right, doc_id=doc["id"])

graph.nodes(data=True), graph.edges(data=True)

In [ ]:
communities = list(nx.algorithms.community.greedy_modularity_communities(graph))

community_summaries = []
for index, community in enumerate(communities, start=1):
    doc_ids = sorted(set().union(*(graph.nodes[node]["docs"] for node in community)))
    texts = [doc["text"] for doc in docs if doc["id"] in doc_ids]
    community_summaries.append({
        "community": index,
        "entities": sorted(community),
        "doc_ids": doc_ids,
        "summary": " ".join(texts),
    })

community_summaries

In [ ]:
def local_graph_search(entity, radius=1):
    if entity not in graph:
        return []
    nodes = nx.single_source_shortest_path_length(graph, entity, cutoff=radius).keys()
    doc_ids = sorted(set().union(*(graph.nodes[node]["docs"] for node in nodes)))
    return [doc for doc in docs if doc["id"] in doc_ids]

local_graph_search("GraphRAG")

In [ ]:
def global_graph_context():
    return "\n\n".join(
        f"Community {row['community']}: entities={row['entities']}\n{row['summary']}"
        for row in community_summaries
    )

print(global_graph_context())

A production GraphRAG notebook can replace the regex extractor with LLM extraction, add source-grounded triples, persist to Neo4j or NetworkX, and compare graph local/global retrieval against baseline vector RAG using notebook 19.